In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
def get_json(url: str) -> dict:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()

In [0]:
def fetch_all(first_url: str) -> list:
    records = []
    url = first_url

    while url:
        page = get_json(url)
        records.extend(page['data'])
        url = urljoin(url, page['next']) if page['next'] else None

    return records

In [0]:
def save_json(records: list, path: str) -> None:
    with open(path, 'w') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

In [0]:
def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here
    
    with open(path, 'wb') as f: # 'wb' = write bytes, needed
        f.write(response.content) # .content = the file's bytes, unchanged
    
    return len(response.content)

In [0]:
def extract(raw_dir: str) -> dict:
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(raw_dir, 'orders.json'))

    for filename in ['products.csv', 'teams.csv', 'matches.parquet']:
        download_file(FILES_URL + filename, os.path.join(raw_dir, filename))

    orders_df = pd.json_normalize(orders, sep='_')
    orders_df = orders_df.rename(columns={
        'customer_name': 'customer',
        'customer_city': 'city'
    })

    return {
        'orders': orders_df,
        'products': pd.read_csv(os.path.join(raw_dir, 'products.csv')),
        'teams': pd.read_csv(os.path.join(raw_dir, 'teams.csv')),
        'matches': pd.read_parquet(os.path.join(raw_dir, 'matches.parquet'))
    }


In [0]:
def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates('product_id', keep='first').copy()

    df['team_code'] = df['team_code'].str.strip().str.upper()
    df['category'] = df['category'].str.strip().str.lower()

    price = df['price'].astype(str).str.strip()
    price = price.str.replace(' RSD', '', regex=False)
    price = price.str.replace(',', '.', regex=False)
    df['price'] = pd.to_numeric(price)

    df['active'] = df['active'].astype(str).str.strip().str.lower() == 'yes'

    return df

In [0]:
def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ['order_id', 'customer', 'city', 'payment']:
        df[col] = df[col].str.strip()

    df['order_id'] = df['order_id'].str.upper()
    df['customer'] = df['customer'].str.title()
    df['city'] = df['city'].str.title().fillna('Unknown')

    return df

In [0]:
def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    is_duplicate = df.duplicated('order_id', keep="first")

    return df[~is_duplicate].copy, df[is_duplicate].copy()

In [0]:
def validate_order(row: dict, product_ids: set) -> str:
    quantity = pd.to_numeric(row['quantity'], errors='coerce')
    price = pd.to_numeric(row['unit_price'], errors='coerce')

    if not row['order_id'].startswith('NSS-'):
        return 'invalid order'
    if pd.isna(row['quantity']) or str(row['quantity']).strip() == '':
        return 'missing_quantity'
    if pd.isna(quantity) or quantity <= 0 or quantity % 1 != 0:
        return 'invalid_quantity'
    if pd.isna(price) or price <= 0:
        return 'invalid_price'
    if row['product_id'] not in product_ids:
        return 'unknown_product'
    
    return 'OK'

In [0]:
def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['quantity'] = pd.to_numeric(df['quantity']).astype(int)
    df['unit_price'] = pd.to_numeric(df['unit_price']).astype(float)
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['total'] = df['quantity'] * df['unit_price']

    product_info = products[['product_id', 'name', 'team_code', 'category']]
    product_info = product_info.rename(columns={'name': 'product_name'})

    return df.merge(product_info, on='product_id', how='left')

In [0]:
def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    df = clean_orders(df)
    uniques, duplicates = remove_duplicate_orders(df)
    duplicates['reason'] = 'duplicate'

    product_ids = set(products['product_id'])
    reasons = unique.apply(lambda row: validate_order(row, product_ids), axis=1)

    valid = transform_orders(unique[reasons == 'OK'], products)

    rejected = unique[reasons != 'OK'].copy()
    rejected['reason'] = reasons[reasons != 'OK']
    rejected = pd.concat([rejected, duplicates], ignore_index=True)

    return valid, rejected

In [0]:
def team_results(matches: pd.DataFrame, teams: pd.DataFrame) -> pd.DataFrame:
    played = matches.loc[matches['status'] == 'finished'].copy()
    played['kickoff_ts'] = pd.to_datetime(played['kickoff_ts'])

    codes = dict(zip(teams['team_id'], teams['code']))

    home = pd.DataFrame ({
        'match_id': played['match_id'],
        'match_date': played['kickoff_ts'].normalize(),
        'team_code': played['home_team_id'].map(codes),
        'opponent_code': played['away_team_id'].map(codes),
        'goals_for': played['home_goals'],
        'goals_against': played['away_goals']
    })

    away = pd.DataFrame({
        'match_id': played['match_id'],
        'match_date': played['kickoff_ts'].normalize(),
        'team_code': played['away_team_id'].map(codes),
        'opponent_code': played['home_team_id'].map(codes),
        'goals_for': played['away_goals'],
        'goals_against': played['home_goals']
    })

    for df in [home, away]:
        df['result'] = 'D'
        df.loc[df['goals_for'] > df['goals_against'], 'result'] = 'W'
        df.loc[df['goals_for'] < df['goals_against'], 'result'] = 'L'
        df['points'] = df['result'].map({'W': 3, 'D': 1, 'L': 0})
    
    return pd.concat([home, away], ignore_index=True)

In [0]:
def league_table(results: pd.DataFrame) -> pd.DataFrame:
    df = results.copy()

    df['won'] = (df['result'] == 'W').astype(int)
    df['drawn'] = (df['result'] == 'D').astype(int)
    df['lost'] = (df['result'] == 'L').astype(int)

    table = df.groupby('team_code').agg(
        played = ('match_id', 'count'),
        won = ('won', 'sum'),
        drawn = ('drawn', 'sum'),
        lost = ('lost', 'sum'),
        goals_for = ('goals_for', 'sum'),
        goals_against = ('goals_against', 'sum'),
        points = ('points', 'sum')
    ).reset_index()

    table['goal_diff'] = table['goals_for'] - table['goals_against']

    table = table.sort_values(['points', 'goal_diff', 'goals_for'], ascending=False).reset_index(drop=True)

    table.insert(0, 'position', table.index + 1)

    return table[['position', 'team_code', 'played', 'won', 'drawn', 'lost', 'goals_for', 'goals_against', 'goal_diff', 'points']]



In [0]:
#14

In [0]:
#15

In [0]:
#16

In [0]:
def load(outputs: dict, processed_dir: str) -> dict:
    written = {}
    for name, df in outputs.items():
    path = os.path.join(processed_dir, name)
    if name.endswith('.parquet'):
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)
    written[name] = len(df)
    return written

In [0]:
def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    results = team_results(data['matches'], data['teams'])
    windows = match_windows(results)
    outputs = {
        'clean_orders.parquet': valid,
        'rejected_orders.csv': rejected,
        'products.csv': products,
        'league_table.csv': league_table(results),
        'revenue_by_team.csv': revenue_by(valid.dropna(subset=['team_code']), 'team_code'),
        'revenue_by_category.csv': revenue_by(valid, 'category'),
        'sales_after_results.csv': sales_after_results(valid, windows),
    }
    written = load(outputs, processed_dir)
    summary = {
        'total_rows': len(data['orders']),
        'duplicates': int((rejected['reason'] == 'duplicate').sum()),
        'valid_rows': len(valid),
        'rejected_rows': len(rejected),
        'total_revenue': round(float(valid['total'].sum()), 2),
        'files_written': len(written) + 1,
    }
    pd.DataFrame(list(summary.items()), columns=['metric', 'value']).to_csv(
        os.path.join(processed_dir, 'summary.csv'), index=False)
    return summary